# Day 2: Ingestion and chunking

Study guide: `course/week-1/day-02-ingestion-chunking/README.md`

Goal: turn PDFs into clean, overlapping chunks that each carry their source and page number.

**Setup:** reload edited `.py` files automatically before each cell runs. Run this once after every kernel restart.

In [1]:
%load_ext autoreload
%autoreload 2

## Step 1: Raw PDF text

**Question:** what does a PDF's text really look like when a program reads it?

In [2]:
import pymupdf   # the PDF library (older tutorials call it "fitz")

from research_rag.paths import PAPERS_DIR   # absolute path to data/papers, works from any folder

pdf_path = PAPERS_DIR / "2106.09685.pdf"     # the LoRA paper

document = pymupdf.open(pdf_path)
print("Pages:", len(document))

page_one = document[0]                       # pages are numbered from 0, like a Python list
print(page_one.get_text())

print("=" * 80)

page_three = document[2]                     # the third page is index 2
print(page_three.get_text()[:1500])          # first 1,500 characters

Pages: 26
LORA: LOW-RANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗
Yelong Shen∗
Phillip Wallis
Zeyuan Allen-Zhu
Yuanzhi Li
Shean Wang
Lu Wang
Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwallis, zeyuana,
yuanzhil, swang, luw, wzchen}@microsoft.com
yuanzhil@andrew.cmu.edu
(Version 2)
ABSTRACT
An important paradigm of natural language processing consists of large-scale pre-
training on general domain data and adaptation to particular tasks or domains. As
we pre-train larger models, full ﬁne-tuning, which retrains all model parameters,
becomes less feasible. Using GPT-3 175B as an example – deploying indepen-
dent instances of ﬁne-tuned models, each with 175B parameters, is prohibitively
expensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-
trained model weights and injects trainable rank decomposition matrices into each
layer of the Transformer architecture, greatly reducing the number of trainable pa-
rameters for downstream tasks. Compared to GPT-3 

### Findings
- LoRA paper: 26 pages. Reading order on page 1 is correct (title, authors, abstract).
- Hidden text: the arXiv watermark `arXiv:2106.09685v2 [cs.CL] 16 Oct 2021` (sideways margin text).
- Broken words: `pre-`/`training`, `indepen-`/`dent`; also a line break after every line.
- Ligatures: `ﬁne-tuning` uses a single `ﬁ` character, so a search for "fine-tuning" wouldn't match. Fix: `unicodedata.normalize("NFKC", text)`.
- Noise: lone page numbers, footnotes mixed into text (`parameters.1`), author emails.
- Equations come out scrambled (Σ becomes `X`). Leaving these for now as a known issue.

## Step 2: Data models

`Page` and `Chunk` in `src/research_rag/ingestion/models.py`. Pydantic rejects bad data straight away: the second `Page` below fails on purpose.

In [3]:
from research_rag.ingestion.models import Page, Chunk

page = Page(source="2106.09685.pdf", page_number=1, text="Hello")
print(page)

Page(source="2106.09685.pdf", page_number="first", text="Hello") 

source='2106.09685.pdf' page_number=1 text='Hello'


ValidationError: 1 validation error for Page
page_number
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='first', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing

## Step 3: Loader

`load_pdf()` in `src/research_rag/ingestion/loader.py`: PDF file in, list of `Page` objects out.

In [4]:
from research_rag.ingestion.loader import load_pdf
from research_rag.paths import PAPERS_DIR

pages = load_pdf(PAPERS_DIR / "2106.09685.pdf")
print("Pages:", len(pages))
print(pages[0].source, pages[0].page_number)
print(pages[0].text[:300])

Pages: 26
2106.09685.pdf 1
LORA: LOW-RANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗
Yelong Shen∗
Phillip Wallis
Zeyuan Allen-Zhu
Yuanzhi Li
Shean Wang
Lu Wang
Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwallis, zeyuana,
yuanzhil, swang, luw, wzchen}@microsoft.com
yuanzhil@andrew.cmu.edu
(Version 2)
ABSTRACT
A


Exploring: what are `index` and `pdf_page` inside the loader's loop?

In [5]:
import pymupdf

with pymupdf.open(PAPERS_DIR / "2106.09685.pdf") as document:
    for index, pdf_page in enumerate(document):
        print(index, "|", pdf_page, "|", type(pdf_page))
        if index == 2:      # stop after 3 pages so the output stays short
            break

0 | page 0 of </Users/ashutoshraichurkar/Documents/repos/research-rag/data/papers/2106.09685.pdf, doc# 3> | <class 'pymupdf.Page'>
1 | page 1 of </Users/ashutoshraichurkar/Documents/repos/research-rag/data/papers/2106.09685.pdf, doc# 3> | <class 'pymupdf.Page'>
2 | page 2 of </Users/ashutoshraichurkar/Documents/repos/research-rag/data/papers/2106.09685.pdf, doc# 3> | <class 'pymupdf.Page'>


The loader works on every paper:

In [6]:
for pdf_path in sorted(PAPERS_DIR.glob("*.pdf")):
    pages = load_pdf(pdf_path)
    print(f"{pdf_path.name}: {len(pages)} pages")

1706.03762.pdf: 15 pages
1908.10084.pdf: 11 pages
2004.04906.pdf: 13 pages
2005.11401.pdf: 19 pages
2106.09685.pdf: 26 pages
2201.11903.pdf: 43 pages
2210.03629.pdf: 33 pages
2212.10496.pdf: 11 pages
2305.14314.pdf: 26 pages
2307.03172.pdf: 18 pages
2309.15217.pdf: 8 pages
2310.11511.pdf: 30 pages


## Step 4: Cleaner
_Next: fix ligatures, broken words, line breaks, watermark and page numbers._

In [7]:
from research_rag.ingestion.cleaner import fix_ligatures

before = "full ﬁne-tuning"
after = fix_ligatures(before)

print(repr(before), "| contains 'fine'?", "fine" in before, "| length:", len(before))
print(repr(after),  "| contains 'fine'?", "fine" in after,  "| length:", len(after))

'full ﬁne-tuning' | contains 'fine'? False | length: 15
'full fine-tuning' | contains 'fine'? True | length: 16


In [8]:
raw = pages[0].text
cleaned = fix_ligatures(raw)

print("'fine' in raw text:    ", raw.count("fine"))
print("'fine' in cleaned text:", cleaned.count("fine"))

'fine' in raw text:     0
'fine' in cleaned text: 0


In [9]:
from research_rag.ingestion.cleaner import join_hyphenated_words

samples = [
    "deploying indepen-\ndent instances",
    "large-scale pre-\ntraining",
    "compute- and memory-efficient",
]
for sample in samples:
    print(repr(sample), "→", repr(join_hyphenated_words(sample)))

'deploying indepen-\ndent instances' → 'deploying independent instances'
'large-scale pre-\ntraining' → 'large-scale pretraining'
'compute- and memory-efficient' → 'compute- and memory-efficient'


In [10]:
from research_rag.ingestion.cleaner import remove_noise_lines

sample = """operational efficiency when deployed.
1
arXiv:2106.09685v2  [cs.CL]  16 Oct 2021
GPT-3 175B has 175 billion parameters
3
AREN'T EXISTING SOLUTIONS GOOD ENOUGH?"""

print(remove_noise_lines(sample))

operational efficiency when deployed.
1

GPT-3 175B has 175 billion parameters
3
AREN'T EXISTING SOLUTIONS GOOD ENOUGH?


In [11]:
from research_rag.ingestion.cleaner import remove_noise_lines

print(remove_noise_lines(sample))
print("-" * 40)

removed = 0
for pdf_path in sorted(PAPERS_DIR.glob("*.pdf")):
    for page in load_pdf(pdf_path):
        before = len(page.text.strip().split("\n"))
        after = len(remove_noise_lines(page.text).split("\n"))
        removed += before - after
print("Lines removed across all papers:", removed)

operational efficiency when deployed.
1

GPT-3 175B has 175 billion parameters
3
AREN'T EXISTING SOLUTIONS GOOD ENOUGH?
----------------------------------------
Lines removed across all papers: 205


In [12]:
from research_rag.ingestion.cleaner import clean_text

lora_pages = load_pdf(PAPERS_DIR / "2106.09685.pdf")
raw = lora_pages[0].text

print("RAW:")
print(raw[-600:])          # the end of page 1: footnotes, page number, watermark
print()
print("CLEAN:")
print(clean_text(raw)[-450:])

RAW:
ome parameters or
learning external modules for new tasks. This way, we only need
to store and load a small number of task-speciﬁc parameters in ad-
dition to the pre-trained model for each task, greatly boosting the
operational efﬁciency when deployed. However, existing techniques
∗Equal contribution.
0Compared to V1, this draft includes better baselines, experiments on GLUE, and more on adapter latency.
1While GPT-3 175B achieves non-trivial performance with few-shot learning, ﬁne-tuning boosts its perfor-
mance signiﬁcantly as shown in Appendix A.
1
arXiv:2106.09685v2  [cs.CL]  16 Oct 2021


CLEAN:
l number of task-specific parameters in addition to the pre-trained model for each task, greatly boosting the operational efficiency when deployed. However, existing techniques ∗Equal contribution. 0Compared to V1, this draft includes better baselines, experiments on GLUE, and more on adapter latency. 1While GPT-3 175B achieves non-trivial performance with few-shot learning, fine-tun

## Step 5: Chunker
_Filled in when we get here._

In [13]:
from research_rag.ingestion.chunker import split_into_windows

numbers = list(range(10))          # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

print("size 4, overlap 1:")
for window in split_into_windows(numbers, size=4, overlap=1):
    print(window)

print("size 4, overlap 0:")
for window in split_into_windows(numbers, size=4, overlap=0):
    print(window)

size 4, overlap 1:
[0, 1, 2, 3]
[3, 4, 5, 6]
[6, 7, 8, 9]
size 4, overlap 0:
[0, 1, 2, 3]
[4, 5, 6, 7]
[8, 9]


In [14]:
from research_rag.ingestion.chunker import chunk_pages

lora_pages = load_pdf(PAPERS_DIR / "2106.09685.pdf")
chunks = chunk_pages(lora_pages, size=200, overlap=40)

print("Chunks:", len(chunks))
print()
for chunk in chunks[:2]:
    print(chunk.id, "| page", chunk.page_number, "|", chunk.token_count, "tokens")
    print(chunk.text)
    print()
print("Last chunk:", chunks[-1].id, "| page", chunks[-1].page_number, "|", chunks[-1].token_count, "tokens")

Chunks: 153

2106.09685.pdf::0000 | page 1 | 200 tokens
LORA: LOW-RANK ADAPTATION OF LARGE LANGUAGE MODELS Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu Yuanzhi Li Shean Wang Lu Wang Weizhu Chen Microsoft Corporation {edwardhu, yeshe, phwallis, zeyuana, yuanzhil, swang, luw, wzchen}@microsoft.com yuanzhil@andrew.cmu.edu (Version 2) ABSTRACT An important paradigm of natural language processing consists of large-scale pretraining on general domain data and adaptation to particular tasks or domains. As we pre-train larger models, full fine-tuning, which retrains all model parameters, becomes less feasible. Using GPT-3 175B as an example – deploying independent instances of fine-tuned models, each with 175B parameters, is prohibitively expensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pretrained model weights and injects trainable rank

2106.09685.pdf::0001 | page 1 | 200 tokens
-tuned models, each with 175B parameters, is prohibitively expensive. We propose L

## Step 7: Inspect the chunks
_Filled in when we get here._